# MarkerTracking3D sparse-tracking exploration

Scroll through `evaluation/markertracking_loading.py`'s sim results (all 8 sparsity/prior conditions x 3 movements + standing) against the dense reference (Nitschke et al. 2023 IK/ID, `_DataStructInverse`) and the raw measured GRF.

**Alignment:** a sim's collocation nodes are a literal contiguous slice of the reference's own 175 Hz sample index -- no resampling (see `setup_motion_marker.m`'s `trackingData.trimData(...)`, and `markertracking_loading.NNODES_EARLIER`/`_tracked_window`). So each plot below shows the full reference trial on its own real sample-index x-axis, with the sim overlaid at its correct, exact position (`tracked_start:tracked_end`) rather than an independently-normalized fraction axis. `standing` has no such window (a static pose, tracked by `run_standing_marker`, not a motion) -- its sim value is plotted as a single point instead.

**Units:** the measured reference's GRF is stored in `BW%` (0-260ish), the sim's in `BW` (0-2.6) -- both are converted to `BW` before plotting.

In [ ]:
import sys; sys.path.insert(0, 'evaluation')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact

import markertracking_loading as mtl

sims = mtl.load_markertracking_sims()
reference = mtl.load_markertracking_reference()
sim_joints = mtl.load_markertracking_sim_joints()
reference_joints = mtl.load_markertracking_reference_joints()
sims[['condition', 'is_prior', 'movement', 'trial', 'converged', 'tracked_markers']]

In [ ]:
JOINTS = ['hip_flexion', 'knee_angle', 'ankle_angle']
GRF_COMPONENTS = ['GRF_x', 'GRF_y', 'GRF_z']
SIDES = ['r', 'l']


def _lookup(vars_df, type_, name):
    match = vars_df[(vars_df.type == type_) & (vars_df.name == name)]
    return match.iloc[0] if len(match) else None


def _series(row, column):
    """Extract `row[column]` (`sim` or `mean`) as an array, converting BW% ->
    BW if that's this row's unit -- the measured reference stores GRF in BW%,
    the sim in BW, and plotting them unconverted is off by 100x."""
    arr = np.atleast_1d(np.asarray(row[column], dtype=float))
    return arr * 0.01 if row['unit'] == 'BW%' else arr


def _plot_at(ax, x0, series, **kwargs):
    if len(series) > 1:
        ax.plot(x0 + np.arange(len(series)), series, **kwargs)
    else:
        ax.scatter([x0], series, **kwargs)


def plot_kinematics(sim_vars, inverse_vars, tracked_start, condition, movement, converged):
    fig, axes = plt.subplots(len(JOINTS), len(SIDES), figsize=(10, 7), sharex=True)
    for i, joint in enumerate(JOINTS):
        for k, side in enumerate(SIDES):
            ax = axes[i, k]
            name = f'{joint}_{side}'
            if inverse_vars is not None:
                ref = _lookup(inverse_vars, 'angle', name)
                if ref is not None:
                    _plot_at(ax, 0, _series(ref, 'mean'), color='0.6', lw=1, label='reference (IK, full trial)')
            sim = _lookup(sim_vars, 'angle', name)
            if sim is not None:
                x0 = tracked_start if tracked_start is not None else 0
                _plot_at(ax, x0, _series(sim, 'sim'), color='C0', lw=1.5, label='sim')
            ax.set_title(name, fontsize=9)
            ax.spines[['top', 'right']].set_visible(False)
    axes[0, 0].legend(fontsize=8, frameon=False)
    for ax in axes[-1]:
        ax.set_xlabel('sample (175 Hz)')
    for ax in axes[:, 0]:
        ax.set_ylabel('deg')
    fig.suptitle(f'{condition} / {movement} -- kinematics (converged={converged})')
    fig.tight_layout()


def plot_grf(sim_vars, measured_vars, tracked_start, condition, movement, converged):
    fig, axes = plt.subplots(len(GRF_COMPONENTS), len(SIDES), figsize=(10, 7), sharex=True)
    for i, comp in enumerate(GRF_COMPONENTS):
        for k, side in enumerate(SIDES):
            ax = axes[i, k]
            name = f'{comp}_{side}'
            if measured_vars is not None:
                meas = _lookup(measured_vars, 'GRF', name)
                if meas is not None:
                    _plot_at(ax, 0, _series(meas, 'mean'), color='0.6', lw=1, label='measured (full trial)')
            sim = _lookup(sim_vars, 'GRF', name)
            if sim is not None:
                x0 = tracked_start if tracked_start is not None else 0
                s = _series(sim, 'sim')
                _plot_at(ax, x0, s, color='C1', lw=1.5, label='sim')
                tracked = _series(sim, 'mean')
                if len(tracked) == len(s):
                    _plot_at(ax, x0, tracked, color='k', lw=1.2, ls='--', label='tracked target')
            ax.set_title(name, fontsize=9)
            ax.spines[['top', 'right']].set_visible(False)
    axes[0, 0].legend(fontsize=8, frameon=False)
    for ax in axes[-1]:
        ax.set_xlabel('sample (175 Hz)')
    for ax in axes[:, 0]:
        ax.set_ylabel('BW')
    fig.suptitle(f'{condition} / {movement} -- GRF (converged={converged})')
    fig.tight_layout()

## Scroll through trials

Pick a `condition / movement` combo below; both plots (kinematics, GRF) redraw for it.

In [ ]:
labels = [f'{c} / {m} ({t})' for c, m, t in zip(sims.condition, sims.movement, sims.trial)]
label_to_idx = dict(zip(labels, sims.index))


def show(label):
    row = sims.loc[label_to_idx[label]]
    ref_rows = reference[reference.movement == row.movement]
    ref_row = ref_rows.iloc[0] if len(ref_rows) else None
    inverse_vars = ref_row['inverse'] if ref_row is not None else None
    measured_vars = ref_row['measured'] if ref_row is not None else None
    tracked_start = ref_row['tracked_start'] if ref_row is not None else None
    tracked_start = None if tracked_start is None or np.isnan(tracked_start) else int(tracked_start)
    if not row.converged:
        print(f'WARNING: {label} did not converge -- treat with caution')
    plot_kinematics(row.variables, inverse_vars, tracked_start, row.condition, row.movement, row.converged)
    plot_grf(row.variables, measured_vars, tracked_start, row.condition, row.movement, row.converged)
    plt.show()


interact(show, label=labels);

interactive(children=(Dropdown(description='label', options=('standing / standing (trial0002)', 'normal / stra…

In [ ]:
# Plot aggregate angle, GRF, and MPJPE RMSE/error for each running condition.
# Each subplot compares simulations with and without the prior.
# each subplot has 4 x-axis ticks for the 4 sparsity levels

# Densest -> sparsest by tracked-marker count (42/11/8/4) -- see the `tracked_markers`
# column above.
BASE_CONDITIONS = ['normal', 'sparse_knee_ankle_pelvis', 'sparse_ankle_hand', 'sparse_ankle']


def _channel_rmse(sim_vars, ref_vars, type_, tracked_start, tracked_end):
    """RMSE per (type_, name) channel present in both `sim_vars` and `ref_vars`,
    `ref_vars` sliced to the sim's own tracked window (see the alignment note
    above) before comparing -- no resampling needed, both are the same 175 Hz
    sample index."""
    errs = []
    for name in sim_vars[sim_vars.type == type_]['name'].unique():
        sim, ref = _lookup(sim_vars, type_, name), _lookup(ref_vars, type_, name)
        if sim is None or ref is None:
            continue
        s = _series(sim, 'sim')
        r = _series(ref, 'mean')[tracked_start:tracked_end]
        if len(r) != len(s) or len(s) == 0:
            continue
        errs.append(np.sqrt(np.nanmean((s - r) ** 2)))
    return errs


def _mpjpe(sim_joint_row, ref_joint_row, tracked_start, tracked_end):
    """Mean per-joint position error (mm) between a sim trial's FK joint centers
    and the reference's own FK joint centers (see markertracking_loading.py's
    joint-center section for why the reference needs FK too -- there is no
    direct mocap ground truth for joint CENTERS, only markers). `ref` is sliced
    to the sim's tracked window first, same alignment as `_channel_rmse`; joined
    on joint name defensively even though both sides come from the same model's
    joint table in the same order."""
    ref_names = list(ref_joint_row.joint_names)
    ref_pos = ref_joint_row.joint_pos[tracked_start:tracked_end]
    sim_pos = sim_joint_row.joint_pos
    n = min(len(sim_pos), len(ref_pos))
    if n == 0:
        return np.array([])
    dists = []
    for j, name in enumerate(sim_joint_row.joint_names):
        if name not in ref_names:
            continue
        r = ref_names.index(name)
        d = np.linalg.norm(sim_pos[:n, j, :] - ref_pos[:n, r, :], axis=-1)
        dists.append(d)
    return np.concatenate(dists) * 1000 if dists else np.array([])  # m -> mm


# Non-converged trials (see the warning banners while scrolling above) are
# dropped -- a failed optimization's RMSE isn't a meaningful sparsity data
# point. `standing` is excluded too: it has no reference window (a static
# pose, not a tracked motion -- see `_tracked_window`).
rows = []
for _, row in sims[(sims.movement != 'standing') & sims.converged].iterrows():
    ref_rows = reference[reference.movement == row.movement]
    if not len(ref_rows):
        continue
    ref_row = ref_rows.iloc[0]
    start, end = int(ref_row.tracked_start), int(ref_row.tracked_end)
    base_condition = row.condition[:-len('_prior')] if row.is_prior else row.condition
    angle_errs = _channel_rmse(row.variables, ref_row['inverse'], 'angle', start, end)
    grf_errs = _channel_rmse(row.variables, ref_row['measured'], 'GRF', start, end)
    prior_label = 'prior' if row.is_prior else 'no prior'
    rows.append({'base_condition': base_condition, 'prior_label': prior_label, 'movement': row.movement,
                 'metric': 'Angle RMSE [deg]', 'rmse': np.mean(angle_errs) if angle_errs else np.nan})
    rows.append({'base_condition': base_condition, 'prior_label': prior_label, 'movement': row.movement,
                 'metric': 'GRF RMSE [BW]', 'rmse': np.mean(grf_errs) if grf_errs else np.nan})

    sim_joint_rows = sim_joints[(sim_joints.condition == row.condition) & (sim_joints.movement == row.movement)]
    ref_joint_rows = reference_joints[reference_joints.movement == row.movement]
    if len(sim_joint_rows) and len(ref_joint_rows):
        mpjpe_dists = _mpjpe(sim_joint_rows.iloc[0], ref_joint_rows.iloc[0], start, end)
    else:
        mpjpe_dists = np.array([])
    rows.append({'base_condition': base_condition, 'prior_label': prior_label, 'movement': row.movement,
                 'metric': 'MPJPE [mm]', 'rmse': mpjpe_dists.mean() if len(mpjpe_dists) else np.nan})
rmse_long = pd.DataFrame(rows)

METRIC_ORDER = ['Angle RMSE [deg]', 'GRF RMSE [BW]', 'MPJPE [mm]']
g = sns.catplot(
    data=rmse_long, kind='bar', hue='prior_label', y='rmse', data='movement',
    row='metric', col='base_condition',
    row_order=METRIC_ORDER, col_order=BASE_CONDITIONS,
    sharey='row', height=3, aspect=0.9, legend_out=True,
)
for col_i, condition in enumerate(BASE_CONDITIONS):
    g.axes[0, col_i].set_title(condition)
    g.axes[1, col_i].set_title('')
    g.axes[2, col_i].set_title('')
g.set_xlabels('')
for row_i, ylabel in enumerate(METRIC_ORDER):
    g.axes[row_i, 0].set_ylabel(ylabel)
g.figure.suptitle('Per-movement RMSE/MPJPE vs reference, by sparsity level and prior', y=1.02)

NameError: name 'sim_joints' is not defined

In [ ]:
row.variables

,type,name,unit,mean,var,segment,direction_x,direction_y,direction_z,position_x,position_y,position_z,sim
0,translation,pelvis_tx,m,[],[],[],NaN,NaN,NaN,NaN,NaN,NaN,"[0.3958337442614162, 0.39646176832795416, 0.39..."
1,translation,pelvis_ty,m,[],[],[],NaN,NaN,NaN,NaN,NaN,NaN,"[0.9246931153485758, 0.9255204609807923, 0.925..."
2,translation,pelvis_tz,m,[],[],[],NaN,NaN,NaN,NaN,NaN,NaN,"[-0.7037183198936511, -0.6951749950438055, -0...."
3,angle,hip_flexion_r,deg,[],[],[],NaN,NaN,NaN,NaN,NaN,NaN,"[20.277793225360202, 20.35906779275338, 20.422..."
4,angle,hip_adduction_r,deg,[],[],[],NaN,NaN,NaN,NaN,NaN,NaN,"[-12.581393324066946, -12.856012510836875, -13..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
123,a,glut_max2_l,[],[],[],[],NaN,NaN,NaN,NaN,NaN,NaN,"[0.0001567925319038338, 0.00014872797644187907..."
124,a,vas_int_r,[],[],[],[],NaN,NaN,NaN,NaN,NaN,NaN,"[0.005672009121839749, 0.00698567560796791, 0...."
125,a,vas_int_l,[],[],[],[],NaN,NaN,NaN,NaN,NaN,NaN,"[0.000997316988593775, 0.0008747620963754665, ..."
126,a,vas_med_r,[],[],[],[],NaN,NaN,NaN,NaN,NaN,NaN,"[0.0105114817978136, 0.0125447342846426, 0.012..."
